In diesem Skript werden die ICON-D2-Daten, die eine Auflösung von 2,2 x 2,2 km haben so umgestellt, dass es keine negativen und positiven Windrichtungen gibt, wie aus "u" und "v" hervorgeht, sondern die Windrichtung in Grad angegeben wird.
Anderenfalls versteht der RF-Algorithmus die negativen Werte nämlich als "sehr geringen" Wind und behandelt diese falsch.

In [1]:
import os
from datetime import datetime, timedelta
import numpy as np
import rasterio

**$$\text{Windrichtung} = \left(\text{atan2}(-u, -v) \times \frac{180}{\pi}\right) \bmod 360$$**

In [2]:
base_dir = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (DOM)\New_ICON_D2_Rasters"

u_dir = os.path.join(base_dir, "U_Wind")
v_dir = os.path.join(base_dir, "V_Wind")
out_dir = os.path.join(base_dir, "Wind_Direction")

# Zielordner erstellen, falls er noch nicht existiert
os.makedirs(out_dir, exist_ok=True)

# Zeitraum definieren: 01.05.2024 bis 31.08.2024 (123 Tage)
start_date = datetime(2024, 5, 1)
end_date = datetime(2024, 8, 31)
days_count = (end_date - start_date).days + 1

print(
    f"Starte Batch-Berechnung der Windrichtung für {days_count} Tage..."
)

# =========================================================
# 2. Schleife über jeden einzelnen Tag
# =========================================================
for i in range(days_count):
    current_date = start_date + timedelta(days=i)
    date_str = current_date.strftime(
        "%Y-%m-%d"
    )

    u_file = os.path.join(u_dir, f"ICON_D2_U_{date_str}.tif") #ICON_D2_U_2024-05-01.tif
    v_file = os.path.join(v_dir, f"ICON_D2_V_{date_str}.tif")
    out_file = os.path.join(out_dir, f"wind_dir_{date_str}.tif")

    # Prüfen, ob beide Quelldateien für den Tag existieren
    if not os.path.exists(u_file) or not os.path.exists(v_file):
        print(f"[{date_str}] Übersprungen: Quelldateien fehlen.")
        continue

    try:
        # GeoTIFFs öffnen und Rasterwerte als NumPy-Arrays einlesen
        with rasterio.open(u_file) as src_u, rasterio.open(v_file) as src_v:
            u_data = src_u.read(1)  # Band 1 auslesen
            v_data = src_v.read(1)

            # Metadaten (CRS, Koordinaten, Pixelgröße) vom Original übernehmen
            meta = src_u.meta.copy()

            # Meteorologische Windrichtung in Grad berechnen
            # atan2(-u, -v) liefert den Winkel im Bogenmaß, deg transformiert in Grad, % 360 hält es zwischen 0° und 360°
            wind_dir = np.mod(
                np.degrees(np.arctan2(-u_data, -v_data)), 360.0
            )

            # Datentyp der Metadaten auf Float32 aktualisieren (für präzise Gradzahlen)
            meta.update(
                {
                    "dtype": "float32",
                    "nodata": -9999.0,  # Leere Pixel sauber kennzeichnen
                }
            )

            # Eventuelle NaN-Werte (z.B. außerhalb des Modellgebiets) abfangen
            wind_dir = np.nan_to_num(wind_dir, nan=-9999.0)

            # Neues GeoTIFF schreiben
            with rasterio.open(out_file, "w", **meta) as dst:
                dst.write(wind_dir.astype("float32"), 1)

        if (i + 1) % 10 == 0 or (i + 1) == days_count:
            print(
                f"[{i+1}/{days_count}] Fortschritt: {date_str} erfolgreich verarbeitet."
            )

    except Exception as e:
        print(f"[{date_str}] Fehler bei der Verarbeitung: {e}")

print(
    f"\nAbgeschlossen! Alle Windrichtungs-Raster liegen unter:\n{out_dir}"
)

Starte Batch-Berechnung der Windrichtung für 123 Tage...
[10/123] Fortschritt: 2024-05-10 erfolgreich verarbeitet.
[20/123] Fortschritt: 2024-05-20 erfolgreich verarbeitet.
[30/123] Fortschritt: 2024-05-30 erfolgreich verarbeitet.
[40/123] Fortschritt: 2024-06-09 erfolgreich verarbeitet.
[50/123] Fortschritt: 2024-06-19 erfolgreich verarbeitet.
[60/123] Fortschritt: 2024-06-29 erfolgreich verarbeitet.
[70/123] Fortschritt: 2024-07-09 erfolgreich verarbeitet.
[80/123] Fortschritt: 2024-07-19 erfolgreich verarbeitet.
[90/123] Fortschritt: 2024-07-29 erfolgreich verarbeitet.
[100/123] Fortschritt: 2024-08-08 erfolgreich verarbeitet.
[110/123] Fortschritt: 2024-08-18 erfolgreich verarbeitet.
[120/123] Fortschritt: 2024-08-28 erfolgreich verarbeitet.
[123/123] Fortschritt: 2024-08-31 erfolgreich verarbeitet.

Abgeschlossen! Alle Windrichtungs-Raster liegen unter:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (DOM)\New_ICON_D2_Rasters\Wind_Direction
